# NIfTI scans to training slices

Builds the real-scan dataset used by `train.py` and zips it as `/kaggle/working/data_real.zip`:

```
data_real/Training/<class>/<patientID>_z###.jpg
data_real/Testing/<class>/<patientID>_z###.jpg
```

Each patient goes entirely into Training or Testing (20% of patients are held out), so test accuracy is not inflated by near-identical slices of the same patient. The conversion code lives in `nifti_slices.py` in the repo.

**Before running:** attach `vietanh21/brats-2021-task-1-dataset` and `dukhailly/meningioma` with *Add Input*, and turn **Internet on** in the notebook settings.

**The first code cell does everything** (find datasets, get the code, convert, zip). The cell after it is an optional preview. Each cell works on its own, so a kernel restart or running one cell alone is fine.

In [ ]:
# ---------- Build data_real.zip (this one cell does the whole job) ----------
import glob, os, shutil, subprocess, sys
from IPython.display import FileLink, display

OUT_DIR = "/kaggle/working/data_real"
ZIP_BASE = "/kaggle/working/data_real"          # .zip is added automatically
REPO_DIR = "/kaggle/working/repo"
SCRIPT = f"{REPO_DIR}/nifti_slices.py"
BRANCH = "main"
SLICES_PER_PATIENT = 8
MAX_PATIENTS = 400
EXTRA_ARGS = []   # e.g. ["--notumor_from_tumor_patients"] or ["--pituitary_root", "/kaggle/input/..."]


def find_input(slug):
    """Find an attached dataset folder by its slug, wherever Kaggle mounted it."""
    hits = [d for d in glob.glob("/kaggle/input/**/", recursive=True)
            if os.path.basename(d.rstrip("/")) == slug and d.count("/") <= 7]
    if not hits:
        raise FileNotFoundError(
            f"Dataset '{slug}' is not attached. Add it with 'Add Input' on the right, then re-run.")
    return sorted(hits, key=len)[0].rstrip("/")


def run(cmd):
    """Run a shell command, show its output live, and stop if it fails."""
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


# 1. Datasets
GLIOMA_ROOT = find_input("brats-2021-task-1-dataset")
MENINGIOMA_ROOT = find_input("meningioma")
print("glioma:    ", GLIOMA_ROOT)
print("meningioma:", MENINGIOMA_ROOT)

# 2. Code
run(["rm", "-rf", REPO_DIR])
run(["git", "clone", "--depth", "1", "-b", BRANCH,
     "https://github.com/imfonggg/AI-for-Brain-Tumor-Detection.git", REPO_DIR])
run([sys.executable, "-m", "pip", "install", "-q", "nibabel"])

# 3. Convert (takes a while: IXI healthy scans are streamed over the internet)
run(["rm", "-rf", OUT_DIR])
run([sys.executable, "-u", SCRIPT, "--out_dir", OUT_DIR,
     "--glioma_root", GLIOMA_ROOT,
     "--meningioma_root", MENINGIOMA_ROOT,
     "--ixi",
     "--slices_per_patient", str(SLICES_PER_PATIENT),
     "--max_patients", str(MAX_PATIENTS),
     *EXTRA_ARGS])
n_images = len(glob.glob(f"{OUT_DIR}/*/*/*.jpg"))
if n_images == 0:
    raise RuntimeError("No slices were written. Check the log above for the class that found 0 cases.")
print(f"\nOK: {n_images} slices in {OUT_DIR}")

# 4. Zip
zip_path = shutil.make_archive(ZIP_BASE, "zip", OUT_DIR)
print(f"Saved {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
print("Download it from the Output panel on the right (/kaggle/working), or click the link:")
os.chdir("/kaggle/working")
display(FileLink(os.path.basename(zip_path)))

In [ ]:
# ---------- Optional: preview 6 random slices per class ----------
import glob, os, random
import matplotlib.pyplot as plt
from PIL import Image

OUT_DIR = "/kaggle/working/data_real"
if not glob.glob(f"{OUT_DIR}/Training/*/*.jpg"):
    raise RuntimeError(f"No slices in {OUT_DIR} yet: run the cell above first.")

classes = sorted(os.listdir(f"{OUT_DIR}/Training"))
fig, axes = plt.subplots(len(classes), 6, figsize=(15, 2.7 * len(classes)), squeeze=False)
for row, cls in enumerate(classes):
    files = glob.glob(f"{OUT_DIR}/Training/{cls}/*.jpg")
    for col, f in enumerate(random.sample(files, min(6, len(files)))):
        axes[row, col].imshow(Image.open(f), cmap="gray")
        axes[row, col].set_title(f"{cls}\n{os.path.basename(f)[:22]}", fontsize=7)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()